# Alice/Bob без пошагового барьера
Qwen3-4B без thinking: сравнение на 1024 токенах, обучение Qwen3-1.7B на короткой траектории.
Независимые циклы; эксклюзивный доступ к кешу только при публикации.
Kernel: **AsyncLLM (uv)**.


In [1]:
import asyncio
import gc
import os
import re
import sys
import time
from contextlib import aclosing
from pathlib import Path
from tempfile import TemporaryDirectory

from IPython.display import Markdown, display

ROOT = Path.cwd().parents[1] if Path.cwd().name == "parallel_thinking" else Path.cwd()
os.chdir(ROOT)
for path in (ROOT / "python", ROOT / "scripts/async_thoughts", ROOT / "scripts/parallel_thinking"):
    sys.path.insert(0, str(path))
os.environ.update(
    CUDA_DEVICE_ORDER="PCI_BUS_ID",
    CUDA_VISIBLE_DEVICES="1",
    TVM_FFI_CUDA_ARCH_LIST="8.6",
    OMP_NUM_THREADS="8",
)
os.environ["PATH"] = f"{Path(sys.executable).parent}:/usr/local/cuda/bin:" + os.environ["PATH"]

import torch
from async_helpers import PublicationGate, isolated, one_token, recorded_example, together
from async_thoughts.training import DemoRecorder
from hogwild_formatting import MathFormatting
from minisgl.core import SamplingParams
from minisgl.llm import AsyncLLM
from minisgl.shared_cache import AsyncContext
from minisgl.training.replay import Replay
from minisgl.training.sft import evaluate, supervised_loss
from transformers import AutoModelForCausalLM, AutoTokenizer

TEACHER = "models/Qwen3-4B"
STUDENT = "models/Qwen3-1.7B"
WORKERS = ("Alice", "Bob")
PROBLEM = (
    "Three vertices of a cube are $P=(7,12,10)$ , $Q=(8,8,1)$ , and "
    "$R=(11,3,9)$ . What is the surface area of the cube?"
)

/home/uskovev/async_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
async def solve_no_barrier(model, *, max_steps=1024, prompt_ids=None, bob_delay=0.0, sampling=None):
    started = time.perf_counter()
    llm = AsyncLLM(
        str(model),
        dtype=torch.bfloat16,
        memory_ratio=0.65,
        max_running_req=4,
        max_seq_len_override=16384,
        cuda_graph_bs=[1, 2],
        cuda_graph_max_bs=2,
        shared_cuda_graph_max_depth=6,
        shared_cuda_graph_prefill_rows=[32, 256],
        max_prefill_rows=256,
        distributed_addr="tcp://127.0.0.1:0",
    )
    llm = DemoRecorder(llm)
    torch.manual_seed(0)
    blocks = []
    try:
        tokenizer = llm.tokenizer
        encode = lambda text: tokenizer.encode(text, add_special_tokens=False)

        fmt = MathFormatting(
            tokenizer, WORKERS, enable_thinking=False
        )
        for _ in range(6):
            blocks.append(await llm.create_block())
        common, others_header, own_header, alice, bob, answer_block = blocks
        worker_blocks = [alice, bob]
        contexts = [
            AsyncContext(cache_view=[common, others_header, bob, own_header, alice]),
            AsyncContext(cache_view=[common, others_header, alice, own_header, bob]),
        ]
        initial_ids = (
            list(prompt_ids) if prompt_ids is not None else encode(fmt.apply_chat_template(PROBLEM))
        )

        await llm.forward(initial_ids, write_to=common, return_logits=False)
        await together(
            llm.forward(
                encode(fmt.current_step_header), write_to=others_header, return_logits=False
            ),
            llm.forward(
                encode(fmt.current_worker_header), write_to=own_header, return_logits=False
            ),
        )

        prefixes = [
            f"{fmt.get_step_prefix(WORKERS[0], 1)}Hi, I'm {WORKERS[0]}. Here's how we can collaborate",
            f"{fmt.get_step_prefix(WORKERS[1], 1)}Hi, I'm {WORKERS[1]}.",
        ]
        current = [encode(prefix) for prefix in prefixes]

        await together(
            *(
                llm.forward(
                    ids[:-1],
                    cache_view=[common, others_header, own_header, block],
                    return_logits=False,
                )
                for ids, block in zip(current, worker_blocks)
            )
        )
        for ctx, ids in zip(contexts, current):
            ctx.next_input_id = ids[-1]

        history, events = [], []
        step_indices, generated = [1, 1], [0, 0]
        sampled = [[], []]
        forbidden = [
            tokenizer.vocab[t]
            for t in ("<|im_start|>", "<|endoftext|>")
            if t in tokenizer.vocab
        ]

        eos = tokenizer.eos_token_id
        forbidden = [token for token in forbidden if token != eos]
        generation_started = time.perf_counter()

        gate = PublicationGate()
        stopped = False
        progress = []

        async def generate_agent(i):
            nonlocal stopped
            ctx = contexts[i]
            for local_step in range(1, max_steps + 1):
                if stopped:
                    break

                if i == 1 and bob_delay and local_step % 32 == 0:
                    await asyncio.sleep(bob_delay)
                async with gate.forward():
                    token = await one_token(llm, ctx, forbidden, sampling)
                    current[i].append(token)
                    generated[i] += 1
                    sampled[i].append(token)
                    progress.append(tuple(generated))
                if token == eos:
                    stopped = True
                    break
                if fmt.is_end_of_step(current[i]):

                    async with gate.publish():
                        await llm.forward(cache_view=ctx, return_logits=False)
                        block = worker_blocks[i]
                        assert block.token_ids == current[i]
                        await llm.append_block(common, block)
                        history.extend(current[i])
                        assert common.token_ids == initial_ids + history
                        events.append(
                            {
                                "worker": WORKERS[i],
                                "step": step_indices[i],
                                "tokens": len(current[i]),
                                "generated": tuple(generated),
                                "text": tokenizer.decode(current[i]),
                            }
                        )
                        await llm.free_block(block)
                        step_indices[i] += 1
                        current[i] = encode(fmt.get_step_prefix(WORKERS[i], step_indices[i]))
                        await llm.forward(current[i][:-1], cache_view=ctx, return_logits=False)
                        ctx.next_input_id = current[i][-1]
                assert ctx.output_block.token_ids == current[i][:-1]
                assert ctx.next_input_id == current[i][-1]

        await together(generate_agent(0), generate_agent(1))
        iteration = max(generated)

        await together(*(llm.forward(cache_view=ctx, return_logits=False) for ctx in contexts))
        for i, ctx in enumerate(contexts):
            assert ctx.output_block.token_ids == current[i]
        parallel_seconds = time.perf_counter() - generation_started
        transcript = fmt.history_header + fmt.sep + tokenizer.decode(history)
        transcript += fmt.sep + fmt.current_step_header
        transcript += "".join(
            tokenizer.decode(ids) + fmt.incomplete_step + fmt.sep for ids in current
        )

        suffix = fmt.s1_finisher_suffix
        answer_ids = encode(suffix)
        answer_ctx = AsyncContext(
            cache_view=[common, others_header, alice, own_header, bob, answer_block]
        )
        await llm.forward(answer_ids[:-1], cache_view=answer_ctx, return_logits=False)
        answer_ctx.next_input_id = answer_ids[-1]
        answer_tokens = []
        async with aclosing(
            llm.async_generate(
                answer_ctx,
                max_steps=64,
                forbid_ids=forbidden,
                sampling_params=SamplingParams(temperature=0.0, max_tokens=64),
            )
        ) as stream:
            async for token in stream:
                if token == eos:
                    break
                answer_tokens.append(token)
                if fmt.get_final_answer(suffix + tokenizer.decode(answer_tokens)) is not None:
                    break
        answer_text = suffix + tokenizer.decode(answer_tokens)
        result = {
            "model": str(model),
            "problem": PROBLEM,
            "rounds": iteration,
            "generated_tokens": dict(zip(WORKERS, generated)),
            "completed_steps": len(events),
            "merge_events": events,
            "transcript": transcript,
            "answer_text": answer_text,
            "answer": fmt.get_final_answer(answer_text),
            "parallel_seconds": parallel_seconds,
            "total_seconds": time.perf_counter() - started,
        }
        result["generated_text"] = {w: tokenizer.decode(ids) for w, ids in zip(WORKERS, sampled)}
        result["prompt_ids"] = initial_ids
        result["history_ids"] = history
        result["current_ids"] = current
        result["answer_ids"] = answer_tokens
        result["progress"] = progress
        result["max_token_gap"] = max(abs(a - b) for a, b in progress)
        result["example"] = recorded_example(llm)
        return result
    finally:

        try:
            for block in blocks:
                await llm.free_block(block)
        finally:
            await llm.close()
            gc.collect()
            torch.cuda.empty_cache()

def show_result(result):
    print("Answer:", result["answer"],
          "Published blocks:", result["completed_steps"])
    print("Generated:", result["generated_tokens"], "Maximum token gap:", result["max_token_gap"])
    print("Generated role headers:", {
        w: re.findall(r"\*\*(?:Alice|Bob) \[\d+\]:\*\*", text)
        for w, text in result["generated_text"].items()
    })
    print("Generated think tags:", {
        w: re.findall(r"</?think>", text) for w, text in result["generated_text"].items()
    })
    display(Markdown(result["transcript"] + "\n\n" + result["answer_text"]))


In [3]:
normal = isolated(solve_no_barrier, TEACHER)
show_result(normal)


Answer: 294.0 Published blocks: 15
Generated: {'Alice': 1024, 'Bob': 1024} Maximum token gap: 1
Generated role headers: {'Alice': [], 'Bob': []}
Generated think tags: {'Alice': [], 'Bob': []}


### Past steps

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate: I'll start by calculating the distances between the given points to find the edge length of the cube. Then I can use that to find the surface area. Let me calculate the distance between P and Q first.

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three of its vertices. Let me start by calculating the distances between the points P, Q, and R to see if they form edges, face diagonals, or space diagonals of the cube. 

**Bob [2]:**  I'll calculate the distance between Q and R. Then I'll compare the results to determine if they are edges, face diagonals, or space diagonals.

**Bob [3]:**  I'll calculate the distance between Q and R. Let me do that now.

**Alice [2]:**  I'll calculate the distance between P and Q. The distance formula in 3D is sqrt[(x2 - x1)^2 + (y2 - y1)^2 + (z2 - z1)^2]. So for P=(7,12,10) and Q=(8,8,1), the distance is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. That's approximately 9.899.

**Bob [4]:**  I'll calculate the distance between Q and R. Q=(8,8,1) and R=(11,3,9). So that's sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. That's also approximately 9.899.

**Bob [5]:**  I see that the distances between P and Q, and Q and R are both sqrt(98). That suggests that PQ and QR are edges of the cube. But wait, if they are edges, then the distance should be the same. However, the distance between P and R is also sqrt(98), which is the same. That suggests that these three points form an equilateral triangle, which is not possible in a cube unless they are on the same face. But in a cube, the maximum distance between two points on the same face is the face diagonal, which is sqrt(2) times the edge length. So if the edge length is 'a', then the face diagonal is a*sqrt(2), and the space diagonal is a*sqrt(3). 

**Alice [3]:**  I'll calculate the distance between P and R. P=(7,12,10) and R=(11,3,9). That's sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. So all three distances are sqrt(98). That's interesting. So the three points are all the same distance apart. That suggests that they form an equilateral triangle, which is not possible in a cube unless they are on the same face. But in a cube, the maximum distance between two points on the same face is the face diagonal, which is sqrt(2) times the edge length. So if the edge length is 'a', then the face diagonal is a*sqrt(2), and the space diagonal is a*sqrt(3). 

**Alice [4]:**  So if the distance between any two of these points is sqrt(98), then that must be the edge length of the cube. Wait, but in a cube, the edge length is the same for all edges. So if the distance between any two adjacent vertices is the edge length, then the edge length is sqrt(98). But wait, that would mean that the edge length is sqrt(98), which is approximately 9.899. But that seems too large for a cube. Let me check my calculations again.

**Bob [6]:**  I see that all three distances are sqrt(98). That suggests that the three points are equidistant from each other. In a cube, this can only happen if they are vertices of a regular triangle, which is not possible unless they are on the same face. But in a cube, the maximum distance between two points on the same face is the face diagonal, which is a*sqrt(2). So if the distance between any two of these points is a*sqrt(2), then the edge length would be a = sqrt(98)/sqrt(2) = sqrt(49) = 7. That makes more sense. So the edge length is 7. Then the surface area of the cube is 6*a^2 = 6*49 = 294.

**Bob [7]:**  I think I need to clarify: if the three points are vertices of a cube, then the distances between them could be either edges, face diagonals, or space diagonals. But if all three distances are the same, then they must be edges of the cube. However, in a cube, the edges are all the same length, so that would mean that the edge length is sqrt(98). But that seems too large. Alternatively, if the three points are not all edges, but two edges and one face diagonal, then the distances would not all be the same. So I think the edge length is sqrt(98), and the surface area is 6*(sqrt(98))^2 = 6*98 = 588.

**Alice [5]:**  Wait, I think I made a mistake. If the distance between two adjacent vertices is the edge length, then the face diagonal is a*sqrt(2), and the space diagonal is a*sqrt(3). But in this case, the distance between P and Q is sqrt(98), which is the same as the distance between Q and R and between P and R. So if these are edges, then the edge length is sqrt(98), but that would make the face diagonal sqrt(98)*sqrt(2) = sqrt(196) = 14, and the space diagonal would be sqrt(98)*sqrt(3) = sqrt(294). But that doesn't match the distances between other points. Alternatively, if the distance between P and Q is the face diagonal, then the edge length would be sqrt(98)/sqrt(2) = sqrt(49) = 7. That makes more sense. So the edge length is 7, and the surface area is 6*7^2 = 294.

**Bob [8]:**  I think I need to verify this. If the edge length is 7, then the face diagonal is 7*sqrt(2) = 14, and the space diagonal is 7*sqrt(3) = 14.7. But the distance between P and Q is sqrt(98) = 9.899, which is less than 14, so that would mean that the distance between P and Q is not the face diagonal. So that suggests that the edge length is not 7. Alternatively, if the edge length is sqrt(98), then the face diagonal is sqrt(98)*sqrt(2) = sqrt(196) = 14, and the space diagonal is sqrt(98)*sqrt(3) = sqrt(294). But the distance between P and Q is sqrt(98), which is less than 14, so that would mean that the distance between P and Q is not the face diagonal. So this is confusing.

**Alice [6]:**  I think I need to double-check the calculations. The distance between P and Q is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. That's correct. The distance between Q and R is sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. That's also correct. The distance between P and R is sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. So all three distances are the same. That suggests that the three points are equidistant from each other. In a cube, this can only happen if they are vertices of a regular triangle, which is not possible unless they are on the same face. But in a cube, the maximum distance between two points on the same face is the face diagonal, which is a*sqrt(2). So if the distance between any two of these points is a*sqrt(2), then the edge length is a = sqrt(98)/sqrt(2) = sqrt(49) = 7. That makes more sense. So the edge length is 7, and the surface area is 6*7^2 = 294.

**Bob [9]:**  I think I need to think differently. If the three points are not all edges, but two edges and one face diagonal, then the distances would not all be the same. So the only way for all three distances to be the same is if they are all edges. But in a cube, the edges are all the same length, so that would mean that the edge length is sqrt(98). But that seems too large. Alternatively, if the three points are not all edges, but two edges and one face diagonal, then the distances would not all be the same. So I think the edge length is 7, and the surface area is 294.



### Work in progress (others)

**Alice [7]:**  I think I need to clarify this. If the three points are vertices of<...>

**Bob [10]:**  I think I need to confirm the edge length. If the edge<...>





Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}.

## Независимый темп
Задержка Bob на 50 мс каждые 32 токена.


In [4]:
delayed = isolated(solve_no_barrier, TEACHER, bob_delay=0.05)
print("Same trajectory:", delayed["history_ids"] == normal["history_ids"])
assert delayed["max_token_gap"] > 1

for result in (normal, delayed):
    example = result["example"]
    replay = Replay.from_trace(
        example["trace"], query_chunk_size=64, attention_checkpointing=True, cache_plans=True
    )
    print(
        "Trace rows:",
        example["trace"].num_tokens,
        "Targets:",
        sum(t != -100 for t in example["targets"]),
    )
show_result(delayed)

Same trajectory: False
Trace rows: 3295 Targets: 2052


Trace rows: 3302 Targets: 2052
Answer: 294.0 Published blocks: 16
Generated: {'Alice': 1024, 'Bob': 1024} Maximum token gap: 65
Generated role headers: {'Alice': [], 'Bob': []}
Generated think tags: {'Alice': [], 'Bob': []}


### Past steps

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate: I'll start by calculating the distances between the given points to find the edge length of the cube. Then I can use that to find the surface area. Let me calculate the distance between P and Q first.

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three of its vertices. Let me start by calculating the distances between the points P, Q, and R to see if they form edges, face diagonals, or space diagonals of the cube. 

**Bob [2]:**  I'll calculate the distance between Q and R next. Then I'll compare the results to determine if they are edges, face diagonals, or space diagonals.

**Bob [3]:**  I'll calculate the distance between Q and R. Let me do that now.

**Alice [2]:**  I'll calculate the distance between P and Q. The distance formula in 3D is sqrt[(x2 - x1)^2 + (y2 - y1)^2 + (z2 - z1)^2]. So for P=(7,12,10) and Q=(8,8,1), the distance is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. That's approximately 9.899.

**Bob [4]:**  I'll calculate the distance between Q and R. Q=(8,8,1) and R=(11,3,9). So that's sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. That's also approximately 9.899.

**Alice [3]:**  I'll calculate the distance between P and R. P=(7,12,10) and R=(11,3,9). That's sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. So all three distances are sqrt(98). That's interesting. So the distance between any two of these points is sqrt(98). That suggests that these three points are vertices of a regular triangle, but in 3D space. 

**Bob [5]:**  I see that the distances between P and Q, and Q and R are both sqrt(98). That suggests that PQ and QR are edges of the cube or face diagonals. Let me check the distance between P and R. Wait, I just calculated that as well, and it's also sqrt(98). So all three distances are the same. That means these three points form an equilateral triangle with side length sqrt(98). But in a cube, the distance between two vertices can be an edge, face diagonal, or space diagonal. 

**Alice [4]:**  I think these three points form an equilateral triangle, but in a cube, the distance between two vertices can be an edge, face diagonal, or space diagonal. If the distance is sqrt(98), that could be a face diagonal or a space diagonal. Let me check the possible lengths. For a cube with edge length a, the face diagonal is a*sqrt(2), and the space diagonal is a*sqrt(3). So if sqrt(98) is a face diagonal, then a*sqrt(2) = sqrt(98), so a = sqrt(98)/sqrt(2) = sqrt(49) = 7. If it's a space diagonal, then a*sqrt(3) = sqrt(98), so a = sqrt(98)/sqrt(3) = sqrt(98/3), which is not an integer. Since the edge length of a cube is usually an integer, I think it's more likely that sqrt(98) is a face diagonal, so the edge length is 7. Then the surface area is 6*7^2 = 294.

**Bob [6]:**  Let me think. If the distance between any two of these points is sqrt(98), then they could be vertices of a regular triangle. But in a cube, the distance between two vertices can be a, a*sqrt(2), or a*sqrt(3). So if sqrt(98) is a face diagonal, then a*sqrt(2) = sqrt(98), so a = sqrt(98)/sqrt(2) = sqrt(49) = 7. That would make the edge length 7. Then the surface area would be 6*a^2 = 6*49 = 294. But if sqrt(98) is a space diagonal, then a*sqrt(3) = sqrt(98), so a = sqrt(98)/sqrt(3), which is not an integer. So I think the edge length is 7, and the surface area is 294.

**Bob [7]:**  Wait, but I need to confirm that these three points are not on the same face of the cube. If they are on the same face, then the distance between any two of them would be either an edge or a face diagonal. But if they are not on the same face, then the distance between two of them could be a space diagonal. However, since all three distances are the same, they must form an equilateral triangle. In a cube, an equilateral triangle can't be formed by three vertices on the same face. So they must be on different faces. Therefore, the edge length is 7, and the surface area is 294.

**Alice [5]:**  I agree. The edge length is 7, so the surface area is 6*7^2 = 294. But I need to make sure that these three points are not on the same face. If they are on the same face, then the distance between any two of them would be either an edge or a face diagonal. But since all three distances are the same, they must form an equilateral triangle. In a cube, an equilateral triangle can't be formed by three vertices on the same face, because the distances would be either edges or face diagonals, but not all the same. So they must be on different faces. Therefore, the edge length is 7, and the surface area is 294.

**Bob [8]:**  I think that's correct. The edge length is 7, so the surface area is 6*7^2 = 294. I'll double-check the distances again. Between P and Q: sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. Between Q and R: sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. Between P and R: sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. All three distances are the same, so they form an equilateral triangle. Therefore, the edge length is 7, and the surface area is 294.

**Alice [6]:**  I think that's the correct answer. The surface area is 294. But I need to double-check my calculations. Let me verify the distances again. The distance between P and Q is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. Between Q and R: sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. Between P and R: sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. All three distances are the same, so they form an equilateral triangle. Since the distance is sqrt(98), and that's a face diagonal, the edge length is 7. Therefore, the surface area is 6*7^2 = 294.

**Alice [7]:**  I think that's the correct answer. The surface area of the cube is 294. I'll confirm once more. The edge length is 7, so the surface area is 6*7^2 = 294. I think that's the answer.

**Bob [9]:**  I think that's correct. The surface area is 294. I'll check if there's any other way to approach this problem. For example, if the three points are not on the same face, then the distance between them could be a space diagonal. But since all three distances are the same, they must form an equilateral triangle. In a cube, an equilateral triangle can't be formed by three vertices on the same face, so they must be on different faces. Therefore, the edge length is 7, and the surface area is 294.



### Work in progress (others)

**Alice [8]:**  I think that's the correct answer. The surface area is 294. I'll confirm once more. The edge length is 7, so the surface area is 6*7^2 = 294. I think that's the<...>

**Bob [10]:**  I think that's correct. The surface area is 294. I'll check if there's any other way to approach this problem. For example, if the three points are not on the same face, then the distance between them could be a space diagonal. But since all three<...>





Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}.

## Траектория для обучения

384 токена на агента, без thinking, с sampling. Обучаем по фактическим чтениям кеша и позициям из trace. Часть вычислений у агентов дублируется.


In [5]:
teacher = isolated(
    solve_no_barrier, TEACHER, max_steps=384,
    sampling=SamplingParams(temperature=0.7, top_p=0.8, top_k=20, max_tokens=1),
)
show_result(teacher)
assert teacher["answer"] == 294
assert not any(re.search(r"</?think>|\*\*(?:Alice|Bob) \[\d+\]:\*\*", text)
               for text in teacher["generated_text"].values())
example = teacher["example"]


Answer: 294.0 Published blocks: 2
Generated: {'Alice': 384, 'Bob': 384} Maximum token gap: 1
Generated role headers: {'Alice': [], 'Bob': []}
Generated think tags: {'Alice': [], 'Bob': []}


### Past steps

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three of its vertices. Let me start by calculating the distances between the points P, Q, and R to see if they form edges or face diagonals or space diagonals of the cube. I'll use the distance formula: distance = sqrt[(x2 - x1)^2 + (y2 - y1)^2 + (z2 - z1)^2].

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate: I'll start by finding the distances between the given points to figure out the side length of the cube. Once I have that, I can calculate the surface area. Let's see... The distance between P and Q is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt(98). Hmm, that's 7*sqrt(2). Wait, that might be a face diagonal. Let me check the distance between Q and R: sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt(98), same as before. So maybe these are both face diagonals. Then the distance between P and R is sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt(98). Wait, that's also sqrt(98). So all three distances are the same. That's interesting. If all three distances are equal, then the triangle PQR is equilateral. But in a cube, the distance between any two vertices can be an edge, face diagonal, or space diagonal. Since all three distances are equal, they must all be face diagonals. So the side length of the cube would be 7, because the face diagonal of a cube with side length s is s*sqrt(2). So s = 7. Then the surface area is 6*s^2 = 6*49 = 294. But wait, that seems too easy. Let me double-check.



### Work in progress (others)

**Alice [2]:**  I see what you mean. If all three distances are the<...>

**Bob [2]:**  I'm Bob. I'll calculate the distance between Q and R next. Let me do that: sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt(98). That's also 7*sqrt(2). Interesting, so both PQ and QR are face diagonals. What about PR? Let me calculate that: sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt(98). Oh, that's also 7*sqrt(2). Wait, so all three distances are equal. That suggests that the triangle PQR is equilateral with all sides equal to 7*sqrt(2). But in a cube, can three face diagonals form an equilateral triangle? That seems unlikely. Maybe I made a mistake in assuming that these are face diagonals. Let me check the distance between P and Q again. The coordinates are P=(7,12,10), Q=(8,8,1). So x difference is 1, y difference is -4, z difference is -9. Squared: 1 + 16 + 81 = 98. So that's correct. So the distance is 7*sqrt(<...>





Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}.

## Тренировочный пример


In [6]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT)
trace, targets = example["trace"], torch.tensor(example["targets"])
print("PROMPT\n" + tokenizer.decode(teacher["prompt_ids"]))
print("\nTRAINING TARGETS (grouped by output block; prefill/flush are masked)")
for block in sorted({token.block for token in trace.tokens}):
    ids = [
        target
        for token, target in zip(trace.tokens, example["targets"])
        if token.block == block and target != -100
    ]
    if ids:
        print(f"\nBlock {block}, {len(ids)} target tokens:\n{tokenizer.decode(ids)}")
print("\nFirst 8 supervised rows: input → target; position; visible (block, length)")
rows = [
    (token, target) for token, target in zip(trace.tokens, example["targets"]) if target != -100
]
for token, target in rows[:8]:
    print(
        repr(tokenizer.decode([token.token_id])),
        "→",
        repr(tokenizer.decode([target])),
        "position=",
        token.position,
        "view=",
        token.view,
    )
print("Total supervised tokens:", len(rows))

PROMPT
<|im_start|>user
# Collaborative Reasoning

You will collaborate on this problem with another assistant. You will write your thoughts simultaneously with them and collaborate without redundant work. You can collaborate by doing different parts of the problem, double-checking each other's results, trying different approaches, or any other means.

There are 2 assistants, including yourself. You will refer to each other as Alice and Bob.

You will solve the problem together, writing your thoughts in parallel. You will be able to see each other's past and current thoughts as we write them. You will see each other's previous steps as **AssistantName [step]:** <...> .

In the '### Past steps' section, the automated system will gather the thoughts of Alice and Bob as you write them.

After the '### Work in progress (others)' section, you will see the other assistants' unfinished steps. They will write those steps concurrently with you. You will take into account what they are doing. If

## Обучение
Master-веса и AdamW — BF16 на CPU, вычисления — BF16 на GPU.
Replay восстанавливает видимость и позиции; loss — средняя CE по целевым токенам.


In [7]:
torch.manual_seed(7)
replay = Replay.from_trace(
    trace, query_chunk_size=64, attention_checkpointing=True, cache_plans=True
)
student = AutoModelForCausalLM.from_pretrained(STUDENT, dtype=torch.bfloat16).cuda()
student.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

master = {
    name: torch.nn.Parameter(p.detach().cpu().clone()) for name, p in student.named_parameters()
}
optimizer = torch.optim.AdamW(master.values(), lr=3e-5, foreach=False)

def evaluate_student():
    metrics = evaluate(student.eval(), [(replay, targets)], chunk_size=64)
    return {key: metrics[key] for key in ("loss", "accuracy")}

print(
    "Master:",
    next(iter(master.values())).dtype,
    "RoPE frequencies:",
    student.model.rotary_emb.inv_freq.dtype,
)
initial = evaluate_student()
print("Initial:", initial)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 4841.83it/s]

Master: torch.bfloat16 RoPE frequencies: torch.float32


Initial: {'loss': 0.41472020745277405, 'accuracy': 0.866580310880829}


In [8]:
for step in range(1, 81):
    student.train()
    student.zero_grad(set_to_none=True)
    optimizer.zero_grad(set_to_none=True)
    loss = supervised_loss(student, replay, targets, chunk_size=64) / targets.ne(-100).sum()
    loss.backward()
    for name, parameter in student.named_parameters():
        master[name].grad = parameter.grad.detach().cpu()
        parameter.grad = None
    torch.nn.utils.clip_grad_norm_(master.values(), 1.0, error_if_nonfinite=True)
    optimizer.step()
    with torch.no_grad():
        for name, parameter in student.named_parameters():
            parameter.copy_(master[name])
    print(f"Step {step}: loss={loss.item():.6f}")
    if step % 5 == 0:
        final = evaluate_student()
        print("Evaluation:", final)
        if final["accuracy"] == 1.0 and final["loss"] < 0.01:
            break
print("Initial:", initial, "Final:", final)
print("Adam moments:", next(iter(optimizer.state.values()))["exp_avg"].dtype)


Step 1: loss=0.414720


Step 2: loss=0.236091


Step 3: loss=0.102275


Step 4: loss=0.032735


Step 5: loss=0.027016


Evaluation: {'loss': 0.005768286529928446, 'accuracy': 0.9974093264248705}


... (2 outputs left out)

Step 8: loss=0.000996


Step 9: loss=0.000464


Step 10: loss=0.000138


Evaluation: {'loss': 8.450909808743745e-05, 'accuracy': 1.0}
Initial: {'loss': 0.41472020745277405, 'accuracy': 0.866580310880829} Final: {'loss': 8.450909808743745e-05, 'accuracy': 1.0}
Adam moments: torch.bfloat16


## Свободная генерация
Токены и публикации учителя не подставляются.


In [9]:
student.cpu()
gc.collect()
torch.cuda.empty_cache()
with TemporaryDirectory() as checkpoint:
    student.save_pretrained(checkpoint)
    tokenizer.save_pretrained(checkpoint)
    for delay in (0.0, 0.05):
        result = isolated(
            solve_no_barrier, checkpoint, max_steps=teacher["rounds"],
            prompt_ids=teacher["prompt_ids"], bob_delay=delay,
        )
        print("Bob delay:", delay)
        show_result(result)
        print("Same tokens:", all(result[key] == teacher[key]
                                  for key in ("history_ids", "current_ids", "answer_ids")))
        print("Same publication sequence:",
              [(e["worker"], e["text"]) for e in result["merge_events"]]
              == [(e["worker"], e["text"]) for e in teacher["merge_events"]])
        print("Same token interleaving:", result["progress"] == teacher["progress"])
        assert not any(re.search(r"</?think>", text) for text in result["generated_text"].values())


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.74s/it]

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.75s/it]

Bob delay: 0.0
Answer: 294.0 Published blocks: 2
Generated: {'Alice': 384, 'Bob': 384} Maximum token gap: 1
Generated role headers: {'Alice': [], 'Bob': []}
Generated think tags: {'Alice': [], 'Bob': []}


### Past steps

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three of its vertices. Let me start by calculating the distances between the points P, Q, and R to see if they form edges or face diagonals or space diagonals of the cube. I'll use the distance formula: distance = sqrt[(x2 - x1)^2 + (y2 - y1)^2 + (z2 - z1)^2].

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate: I'll start by finding the distances between the given points to figure out the side length of the cube. Once I have that, I can calculate the surface area. Let's see... The distance between P and Q is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt(98). Hmm, that's 7*sqrt(2). Wait, that might be a face diagonal. Let me check the distance between Q and R: sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt(98), same as before. So maybe these are both face diagonals. Then the distance between P and R is sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt(98). Wait, that's also sqrt(98). So all three distances are the same. That's interesting. If all three distances are equal, then the triangle PQR is equilateral. But in a cube, the distance between any two vertices can be an edge, face diagonal, or space diagonal. Since all three distances are equal, they must all be face diagonals. So the side length of the cube would be 7, because the face diagonal of a cube with side length s is s*sqrt(2). So s = 7. Then the surface area is 6*s^2 = 6*49 = 294. But wait, that seems too easy. Let me double-check.



### Work in progress (others)

**Alice [2]:**  I see what you mean. If all three distances are the<...>

**Bob [2]:**  I'm Bob. I'll calculate the distance between Q and R next. Let me do that: sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt(98). That's also 7*sqrt(2). Interesting, so both PQ and QR are face diagonals. What about PR? Let me calculate that: sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt(98). Oh, that's also 7*sqrt(2). Wait, so all three distances are equal. That suggests that the triangle PQR is equilateral with all sides equal to 7*sqrt(2). But in a cube, can three face diagonals form an equilateral triangle? That seems unlikely. Maybe I made a mistake in assuming that these are face diagonals. Let me check the distance between P and Q again. The coordinates are P=(7,12,10), Q=(8,8,1). So x difference is 1, y difference is -4, z difference is -9. Squared: 1 + 16 + 81 = 98. So that's correct. So the distance is 7*sqrt(<...>





Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}.

Same tokens: True
Same publication sequence: True
Same token interleaving: True


Bob delay: 0.05
Answer: 294.0 Published blocks: 2
Generated: {'Alice': 384, 'Bob': 384} Maximum token gap: 43
Generated role headers: {'Alice': [], 'Bob': []}
Generated think tags: {'Alice': [], 'Bob': []}


### Past steps

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three of its vertices. Let me start by calculating the distances between the points P, Q, and R to see if they form edges or face diagonals or space diagonals of the cube. I'll use the distance formula: distance = sqrt[(x2 - x1)^2 + (y2 - y1)^2 + (z2 - z1)^2].

**Bob [2]:**  I'm Bob. I'll calculate the distance between Q and R next. Let me do that: sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt(98). Interesting, that's the same as the distance between Q and P. So maybe these are both face diagonals. Then I'll calculate the distance between P and Q again to confirm. Wait, I already did that. So if both distances are 7*sqrt(2), then the side length of the cube would be 7, because the face diagonal of a cube with side length s is s*sqrt(2). So s = 7. Then the surface area is 6*s^2 = 6*49 = 294. But wait, that seems too easy. Let me double-check.



### Work in progress (others)

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate: I'll start by finding the distances between the given points to figure out the side length of the cube. Once I have that, I can calculate the surface area. Let's see... The distance between P and Q is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt(98). Hmm, that's 7*sqrt(2). Wait, that might be a face diagonal. Let me check the distance between Q and R: sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt(98). That's also 7*sqrt(2). So maybe these are both face diagonals. Then the distance between P and R is sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt(98). Wait, that's also 7*sqrt(2). So all three distances are the same. That's interesting. If all three distances are equal, then the side length would be 7, because the distance between any two vertices that are not on the same face would be 7*sqrt(2). So the side length is 7. Then the surface area is 6*7^2 = 294. But I should double-check this because it seems too easy. Let me calculate the distance between P and Q again. x difference is 1, y difference is -4, z difference is -9. Squared: 1 + 16 + 81 = 98. So that's correct. So<...>

**Bob [3]:**  I'm Bob. I'll calculate the distance between P and Q again to make sure I didn't make a mistake. Let me do that: (8-7)=1, (8-12)=-4, (1-10)=-9. Squared: 1 + 16 + 81 = 98. Yep, that's correct. So the distance is 7*sqrt(2). So the side length is 7. Then the surface area is 6*49 =<...>





Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}.

Same tokens: False
Same publication sequence: False
Same token interleaving: False
